In [11]:
import numpy as np
import ase.io
import ase.units as units
from ase.build import bulk
from ase.constraints import ExpCellFilter
from ase.optimize.precon import PreconLBFGS
from ase.optimize import LBFGSLineSearch
from nglview import show_ase
from matscipy.fracture_mechanics.crack import CubicCrystalCrack
from matscipy.fracture_mechanics.clusters import diamond, set_groups
from matscipy.calculators.manybody.explicit_forms.stillinger_weber import StillingerWeber, Holland_Marder_PRL_80_746_Si
from matscipy.calculators.manybody import Manybody
from matscipy.elasticity import fit_elastic_constants


In [ ]:
# Initialize Stillinger-Weber potential for silicon
calc = Manybody(**StillingerWeber(Holland_Marder_PRL_80_746_Si))

# Define chemical element and lattice parameters
el = 'Si'  # Silicon
# Optimize bulk silicon to determine equilibrium lattice constant
si = bulk(el, cubic=True)
si.calc = calc
ecf = ExpCellFilter(si)
opt = PreconLBFGS(ecf, logfile='logs/bulk_optimization.log')
opt.run(fmax=1e-6, smax=1e-6)  # Tight convergence for lattice constant
a0 = si.cell[0, 0]
ase.io.write('bulk_optimized.cfg', si)  # Save optimized bulk structure

/tmp/ipykernel_2377428/190812157.py:9: FutureWarning: Import ExpCellFilter from ase.filters
  ecf = ExpCellFilter(si)


In [13]:
# Calculate elastic constants for cubic silicon
si = bulk(el, a=a0)
si.calc = calc
C, C_err = fit_elastic_constants(si, symmetry='cubic')
C_11 = C[0, 0] / units.GPa
C_12 = C[0, 1] / units.GPa
C_44 = C[3, 3] / units.GPa
print(f"Elastic constants: C_11 = {C_11:.2f} GPa, C_12 = {C_12:.2f} GPa, C_44 = {C_44:.2f} GPa")

Fitting C_11
Strain array([-0.02, -0.01,  0.  ,  0.01,  0.02])
Stress array([-4.18432088e+00, -2.05307995e+00,  2.76443651e-05,  1.97564777e+00,
        3.87438786e+00]) GPa
Cij (gradient) / GPa    :     201.46145206140287
Error in Cij / GPa      :     2.647085830911924
Correlation coefficient :     0.9997411341719051
Setting C11 (1) to 1.257423 +/- 0.016522


Fitting C_21
Strain array([-0.02, -0.01,  0.  ,  0.01,  0.02])
Stress array([-1.13681364e+00, -5.40617973e-01,  2.76443651e-05,  4.89093693e-01,
        9.30336637e-01]) GPa
Cij (gradient) / GPa    :     51.640122172534795
Error in Cij / GPa      :     1.764430372335353
Correlation coefficient :     0.9982534266249888
Setting C21 (7) to 0.322312 +/- 0.011013


Fitting C_31
Strain array([-0.02, -0.01,  0.  ,  0.01,  0.02])
Stress array([-1.13681364e+00, -5.40617973e-01,  2.76443651e-05,  4.89093693e-01,
        9.30336637e-01]) GPa
Cij (gradient) / GPa    :     51.640122172534866
Error in Cij / GPa      :     1.7644303723352428
Co

In [ ]:
# Function to calculate surface energy with convergence testing
def find_surface_energy(symbol, calc, a0, surface, size=(16, 2, 2), vacuum=20.0, fmax=0.001, unit='0.1J/m^2'):
    """Calculate surface energy for a given surface orientation and system size."""
    from ase.lattice.cubic import Diamond as lattice_builder
    if surface.endswith('111'):
        directions = [[1, 1, 1], [-2, 1, 1], [0, -1, 1]]
    else:
        raise ValueError('Unsupported surface orientation.')
    
    # Create bulk and slab with same number of atoms
    bulk = lattice_builder(directions=directions, size=size, symbol=symbol, latticeconstant=a0, pbc=(1, 1, 1))
    cell = bulk.get_cell()
    cell[0, :] *= 2  # Add vacuum along x-axis (surface normal)
    slab = bulk.copy()
    slab.set_cell(cell)
    
    # Optimize bulk and slab with strict convergence
    bulk.calc = calc
    opt_bulk = LBFGSLineSearch(bulk, logfile=f'logs/bulk_opt_{size}.log')
    opt_bulk.run(fmax=fmax)
    ase.io.write(f'bulk_optimized_{size}.cfg', bulk)
    
    slab.calc = calc
    opt_slab = LBFGSLineSearch(slab, logfile=f'logs/slab_opt_{size}.log')
    opt_slab.run(fmax=fmax)
    ase.io.write(f'slab_optimized_{size}.cfg', slab)
    
    # Calculate surface energy
    Ebulk = bulk.get_potential_energy()
    Eslab = slab.get_potential_energy()
    area = np.linalg.norm(np.cross(slab.get_cell()[1, :], slab.get_cell()[2, :]))
    gamma_ase = (Eslab - Ebulk) / (2 * area)
    
    # Convert to requested units
    if unit == 'ASE':
        return [gamma_ase, 'ase_units']
    else:
        from ase import units
        gamma_SI = (gamma_ase / units.J) * (units.m)**2
        if unit == 'J/m^2':
            return [gamma_SI, 'J/m^2']
        elif unit == '0.1J/m^2':
            return [10 * gamma_SI, '0.1J/m^2']
        else:
            raise ValueError('Unsupported unit of surface energy.')

# Perform convergence test for surface energy
sizes = [(8, 2, 2), (16, 2, 2), (32, 4, 4)]
surface = 'diamond111'
for size in sizes:
    gamma, unit = find_surface_energy(el, calc, a0, surface, size=size, vacuum=20.0, fmax=0.001)
    print(f"Surface energy for size {size}: {gamma:.4f} {unit}")

Surface energy for size (8, 2, 2): 40.8008 0.1J/m^2
Surface energy for size (16, 2, 2): 40.8008 0.1J/m^2
Surface energy for size (32, 4, 4): 40.8008 0.1J/m^2


In [15]:
# Use the largest system size for final surface energy
gamma, unit = find_surface_energy(el, calc, a0, surface, size=(32, 4, 4), vacuum=20.0, fmax=0.001)
# Setup crack system
n = [8, 8, 2]  # Larger system size to reduce finite-size effects
crack_surface = [1, 1, 1]  # (111) surface
crack_front = [1, -1, 0]  # [110] crack front
skin_x, skin_y = 2, 2  # Skin region for group setting
vacuum = 20.0  # Increased vacuum to avoid interactions
fmax = 0.001  # Stricter convergence for crack system

# Create diamond lattice with crack orientation
cryst = diamond(el, a0, n, crack_surface, crack_front)
set_groups(cryst, n, skin_x, skin_y)
ase.io.write('cryst_initial.cfg', cryst)  # Save initial crystal structure

skin_x = 2*a0, skin_y = 2*a0
skin_x = 13.303056984425343, skin_y = 18.81336360839645


In [20]:
# Identify crack tip dynamically (center of the system in x-y plane)
positions = cryst.get_positions()
tip_x0 = np.mean(positions[:, 0])  # Approximate crack tip at center
tip_y0 = np.mean(positions[:, 1])
tip_z0 = np.mean(positions[:, 2])
print(f"Initial crack tip position: ({tip_x0:.2f}, {tip_y0:.2f}, {tip_z0:.2f})")

# Initialize crack using linear elastic fracture mechanics
crk = CubicCrystalCrack(crack_surface, crack_front, C_11, C_12, C_44)
k1g = crk.k1g(gamma)
print(f"Griffith stress intensity factor: k1g = {k1g:.4f} MPa*sqrt(m)")

# Apply LEFM displacements to create crack
crack = cryst.copy()
crack.set_pbc([False, False, True])  # Periodic along crack front (z)
tip_x = crack.cell.diagonal()[0] / 2
tip_y = crack.cell.diagonal()[1] / 2
k1 = 1.0  # Initial stress intensity factor (scaled to k1g)
ux, uy = crk.displacements(crack.positions[:, 0], crack.positions[:, 1], tip_x, tip_y, k1 * k1g)
crack.positions[:, 0] += ux
crack.positions[:, 1] += uy
show_ase(crack)

Initial crack tip position: (26.61, 37.63, 3.84)
Griffith stress intensity factor: k1g = 137.7213 MPa*sqrt(m)


NGLWidget()

In [17]:
# Center the system and ensure sufficient vacuum
oldr = crack[0].position.copy()
crack.center(vacuum=vacuum, axis=0)
crack.center(vacuum=vacuum, axis=1)
tip_x += crack[0].x - oldr[0]
tip_y += crack[0].y - oldr[1]

# Optimize the crack system
crack.calc = calc
opt_crack = LBFGSLineSearch(crack, logfile='crack_opt.log')
opt_crack.run(fmax=fmax)
ase.io.write('crack_optimized.cfg', crack)  # Save final crack configuration

# Visualize the final crack configuration
show_ase(crack)

KeyboardInterrupt: 